# Preprocessing Dữ liệu — RAG Soạn thảo Văn bản Hành chính

Notebook thực hiện **data cleaning** cho cả 3 bộ dữ liệu trước khi đưa vào pipeline RAG.

| Dataset | File gốc | Vấn đề chính | Hành động |
|---------|----------|--------------|----------|
| `legal_dataset` | `legal_dataset.parquet` | 17.6% content null, 6.3% content dup, type nhiễu (98 giá trị) | Drop null/dup, normalize type → 8 nhóm chuẩn, normalize text, tạo doc_id duy nhất |
| `forms_dataset` | `forms_dataset.parquet` | Sạch — chỉ cần normalize text nhẹ | Normalize text, tạo doc_id |
| `forms_examples_dataset` | `forms_examples_dataset.parquet` | Sạch hoàn toàn | Normalize text, serialize cột dict/list, lưu processed version |

> ⚠️ **Chunking** sẽ được thực hiện ở notebook riêng ở bước sau trong pipeline.

---
## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import re
import json
import hashlib
from pathlib import Path

# ── Đường dẫn ─────────────────────────────────────────────────────────────────
DATASET_DIR = Path("../dataset/raw")

INPUT_PATHS = {
    "legal"   : DATASET_DIR / "legal_dataset.parquet",
    "forms"   : DATASET_DIR / "forms_dataset.parquet",
    "examples": DATASET_DIR / "forms_examples_dataset.parquet",
}

OUTPUT_DIR = Path("dataset/processed")

OUTPUT_PATHS = {
    "legal"   : OUTPUT_DIR / "legal_dataset_processed.parquet",
    "forms"   : OUTPUT_DIR / "forms_dataset_processed.parquet",
    "examples": OUTPUT_DIR / "forms_examples_dataset_processed.parquet",
}

# ── Hàm tiện ích ─────────────────────────────────────────────────────────────
def log_step(step: str, before: int, after: int, note: str = ""):
    """In báo cáo số hàng trước/sau mỗi bước cleaning."""
    removed = before - after
    pct     = removed / before * 100 if before > 0 else 0
    tag     = f"  [{step}]  {before:>9,} → {after:>9,}  (xóa {removed:>7,} | {pct:.1f}%)"
    if note:
        tag += f"  ← {note}"
    print(tag)

print("✅ Setup hoàn tất.")
for name, path in INPUT_PATHS.items():
    status = "✅" if path.exists() else "❌ KHÔNG TÌM THẤY"
    print(f"  [{status}] {name}: {path}")

✅ Setup hoàn tất.
  [✅] legal: ../dataset/raw/legal_dataset.parquet
  [✅] forms: ../dataset/raw/forms_dataset.parquet
  [✅] examples: ../dataset/raw/forms_examples_dataset.parquet


---
## 1. Legal Dataset

**Vấn đề phát hiện từ checkData_v2:**
- 90,706 hàng (17.61%) có `content = null` → không thể embed, phải loại bỏ
- 2,433 hàng (0.47%) trùng hoàn toàn (exact duplicate)
- 32,403 hàng (6.29%) có nội dung `content` giống nhau (content duplicate)
- Cột `type` có 98 giá trị unique do bị ngắt dòng, tiếng Anh, nhiễu → cần chuẩn hoá
- Cột `id` là số hiệu văn bản (95% trùng vì nhiều điều khoản cùng văn bản) → **không dùng làm doc_id**

### 1.1. Đọc dữ liệu gốc

In [2]:
legal_df = pd.read_parquet(INPUT_PATHS["legal"], engine="pyarrow")

print(f"Tổng bản ghi : {len(legal_df):,}")
print(f"Cột          : {list(legal_df.columns)}")
print(f"\nKiểu dữ liệu:")
print(legal_df.dtypes)
print(f"\nMissing values:")
print(legal_df.isnull().sum())

Tổng bản ghi : 515,188
Cột          : ['id', 'ministry', 'type', 'name', 'chapter_id', 'chapter_name', 'article', 'content']

Kiểu dữ liệu:
id              str
ministry        str
type            str
name            str
chapter_id      str
chapter_name    str
article         str
content         str
dtype: object

Missing values:
id                350
ministry          757
type               35
name               16
chapter_id          3
chapter_name        0
article           309
content         90706
dtype: int64


### 1.2. Xóa hàng có `content` null / rỗng

> checkData: **17.61%** (~90,706 hàng) — dữ liệu không thể embed, loại bỏ trước tiên.

In [3]:
n0 = len(legal_df)

# Bước 1: Xóa NaN
legal_df = legal_df.dropna(subset=["content"])
n_after_nan = len(legal_df)
log_step("Drop NaN content    ", n0, n_after_nan, "content = null")

# Bước 2: Xóa chuỗi rỗng / chỉ có khoảng trắng
legal_df = legal_df[legal_df["content"].str.strip() != ""]
n1 = len(legal_df)
log_step("Drop empty content  ", n_after_nan, n1, "content = '' hoặc chỉ whitespace")

print(f"\n✅ Sau bước 1.2: còn {n1:,} hàng")

  [Drop NaN content    ]    515,188 →   424,482  (xóa  90,706 | 17.6%)  ← content = null
  [Drop empty content  ]    424,482 →   424,482  (xóa       0 | 0.0%)  ← content = '' hoặc chỉ whitespace

✅ Sau bước 1.2: còn 424,482 hàng


### 1.3. De-duplicate

Hai lượt dedup:
- **Exact duplicate**: toàn bộ hàng giống nhau
- **Content duplicate**: cùng nội dung `content` (khác metadata) — giữ hàng đầu

> checkData: 0.47% exact dup + 6.29% content dup.

In [4]:
n1 = len(legal_df)

# Lượt 1: Exact duplicate
legal_df = legal_df.drop_duplicates(keep="first")
n_exact = len(legal_df)
log_step("Exact duplicate     ", n1, n_exact)

# Lượt 2: Content duplicate
legal_df = legal_df.drop_duplicates(subset=["content"], keep="first")
n2 = len(legal_df)
log_step("Content duplicate   ", n_exact, n2)

print(f"\n✅ Sau bước 1.3: còn {n2:,} hàng  (đã loại {n1 - n2:,} | {(n1-n2)/n1*100:.1f}%)")

  [Exact duplicate     ]    424,482 →   423,844  (xóa     638 | 0.2%)
  [Content duplicate   ]    423,844 →   392,079  (xóa  31,765 | 7.5%)

✅ Sau bước 1.3: còn 392,079 hàng  (đã loại 32,403 | 7.6%)


### 1.4. Chuẩn hoá cột `type` → 8 nhóm

Cột `type` gốc có **98 giá trị unique** do:
- Giá trị bị ngắt dòng giữa chừng ("QUYẾT" / "ĐỊNH" tách thành 2 hàng riêng)
- Lẫn tiếng Anh ("CIRCULAR", "ORDINANCE")
- Có giá trị nhiễu ("none", số,...)

Map về **8 nhóm chuẩn** bằng keyword matching, không nhận ra → `KHAC`.

In [5]:
# Bảng map keyword → nhóm chuẩn
# Key: nhãn chuẩn  |  Value: danh sách keyword (lowercase, dùng 'in' để match substring)
TYPE_KEYWORD_MAP = {
    "THÔNG TƯ"          : ["thông tư liên tịch", "thông tư", "circular"],
    "QUYẾT ĐỊNH"        : ["quyết định", "decision"],
    "NGHỊ QUYẾT"        : ["nghị quyết liên tịch", "nghị quyết của bộ chính trị",
                           "quyết nghị", "nghị quyết", "resolution"],
    "NGHỊ ĐỊNH"         : ["nghị định", "decree"],
    "LUẬT"              : ["bộ luật", "hiến pháp", "luật", "law", "code"],
    "PHÁP LỆNH"         : ["pháp lệnh", "ordinance", "lệnh"],
    "VĂN BẢN LIÊN TỊCH" : ["liên tịch"],
    "VĂN BẢN KHÁC"      : ["quy định", "kế hoạch", "đề án", "báo cáo",
                           "chiến lược", "thông báo", "bảng giá đất",
                           "danh sách", "danh mục", "phụ lục", "chỉ thị"],
}

def normalize_type(raw) -> str:
    """Map giá trị type thô về nhóm chuẩn."""
    if pd.isna(raw):
        return "KHÔNG RÕ"
    # Chuẩn hoá: bỏ khoảng trắng thừa, lowercase để so sánh
    cleaned = re.sub(r"\s+", " ", str(raw).strip()).lower()
    # Ưu tiên match dài hơn trước (đã sắp xếp trong dict)
    for label, keywords in TYPE_KEYWORD_MAP.items():
        for kw in keywords:
            if kw in cleaned:
                return label
    return "KHÁC"

legal_df["type_normalized"] = legal_df["type"].apply(normalize_type)

print(f"type gốc (unique)        : {legal_df['type'].nunique()}")
print(f"type sau normalize (unique): {legal_df['type_normalized'].nunique()}")
print("\nPhân phối type_normalized:")
dist = legal_df["type_normalized"].value_counts()
for label, cnt in dist.items():
    pct = cnt / len(legal_df) * 100
    print(f"  {label:<25} {cnt:>8,}  ({pct:.1f}%)")

type gốc (unique)        : 96
type sau normalize (unique): 9

Phân phối type_normalized:
  THÔNG TƯ                   132,431  (33.8%)
  QUYẾT ĐỊNH                  85,733  (21.9%)
  NGHỊ ĐỊNH                   71,176  (18.2%)
  NGHỊ QUYẾT                  59,827  (15.3%)
  LUẬT                        31,145  (7.9%)
  PHÁP LỆNH                    6,798  (1.7%)
  KHÁC                         4,859  (1.2%)
  VĂN BẢN KHÁC                    83  (0.0%)
  KHÔNG RÕ                        27  (0.0%)


### 1.5. Normalize text

Áp dụng cho các cột chuỗi:
- Strip khoảng trắng đầu/cuối
- Tab → space
- Collapse nhiều space liên tiếp → 1 space (giữ nguyên newline)
- Collapse nhiều newline liên tiếp → tối đa 2 newline

> Không lowercase vì tên luật, tên cơ quan cần giữ chữ hoa.

In [6]:
def normalize_text(text) -> str:
    """Normalize whitespace trong chuỗi, giữ nguyên chữ hoa/thường."""
    if pd.isna(text) or text is None:
        return ""
    text = str(text)
    text = text.replace("\t", " ")           # tab → space
    text = text.replace("\r\n", "\n")        # Windows newline
    text = text.replace("\r", "\n")          # old Mac newline
    text = re.sub(r"[^\S\n]+", " ", text)    # nhiều space → 1 space
    text = re.sub(r"\n{3,}", "\n\n", text)   # nhiều newline → max 2
    return text.strip()

TEXT_COLS = ["content", "name", "article", "chapter_name", "ministry"]
for col in TEXT_COLS:
    if col in legal_df.columns:
        legal_df[col] = legal_df[col].apply(normalize_text)
        print(f"  ✅ Normalized: {col}")

# Xóa lại các hàng content rỗng sau normalize
n_before = len(legal_df)
legal_df = legal_df[legal_df["content"] != ""]
n_after  = len(legal_df)
if n_before != n_after:
    print(f"  ℹ️  Xóa thêm {n_before - n_after:,} hàng content rỗng sau normalize")
else:
    print(f"  ✅ Không có hàng nào bị rỗng thêm sau normalize")

  ✅ Normalized: content
  ✅ Normalized: name
  ✅ Normalized: article
  ✅ Normalized: chapter_name
  ✅ Normalized: ministry
  ✅ Không có hàng nào bị rỗng thêm sau normalize


### 1.6. Tạo `doc_id` duy nhất

**Lý do không dùng cột `id` gốc làm doc_id:**  
Cột `id` là số hiệu văn bản (vd. `30/2020/NĐ-CP`) — nhiều điều khoản cùng văn bản chia sẻ 1 `id` → 95% trùng.  
→ Dùng **hash SHA-256 của `content`** làm doc_id để đảm bảo tính duy nhất tuyệt đối theo nội dung.

In [7]:
def make_legal_doc_id(row) -> str:
    """
    Tạo doc_id duy nhất theo cấp article:
    - Kết hợp id văn bản + article (nếu có) để tạo định danh ngữ nghĩa
    - Fallback SHA-256 của content nếu thiếu thông tin
    """
    doc_no  = str(row.get("id", "") or "").strip()
    article = str(row.get("article", "") or "").strip()

    if doc_no and doc_no.lower() != "nan":
        if article and article.lower() != "nan":
            # Kết hợp số hiệu + tên điều: "30/2020/NĐ-CP__Điều 5"
            base = f"{doc_no}__{article[:80]}"
        else:
            base = doc_no
    else:
        base = ""

    # Luôn thêm hash ngắn của content để tránh collision
    content_hash = hashlib.sha256(
        row["content"].encode("utf-8")
    ).hexdigest()[:10]

    if base:
        # Sanitize: bỏ ký tự đặc biệt không phù hợp làm ID
        base_clean = re.sub(r"[^\w/\-\.]+", "_", base)
        return f"{base_clean}__{content_hash}"
    else:
        return f"sha_{content_hash}"

legal_df["doc_id"] = legal_df.apply(make_legal_doc_id, axis=1)

total  = len(legal_df)
unique = legal_df["doc_id"].nunique()
dup    = total - unique
print(f"Tổng hàng    : {total:,}")
print(f"Unique doc_id: {unique:,}")
if dup > 0:
    print(f"⚠️  Còn {dup:,} doc_id trùng — kiểm tra lại logic")
    # Fallback: thêm thứ tự vào cuối nếu vẫn trùng
    mask = legal_df.duplicated(subset=["doc_id"], keep=False)
    legal_df.loc[mask, "doc_id"] = (
        legal_df.loc[mask, "doc_id"] + "__"
        + legal_df.loc[mask].groupby("doc_id").cumcount().astype(str)
    )
    print(f"  → Đã xử lý fallback. Unique sau: {legal_df['doc_id'].nunique():,}")
else:
    print("✅ Tất cả doc_id unique")

Tổng hàng    : 392,079
Unique doc_id: 392,079
✅ Tất cả doc_id unique


### 1.7. Chọn cột & Lưu Parquet

In [8]:
# Thứ tự cột trong output: doc_id đầu tiên, type_normalized thay thế type gốc
KEEP_COLS = [
    "doc_id",
    "id",              # số hiệu văn bản gốc (giữ lại làm metadata)
    "ministry",
    "type_normalized", # đã chuẩn hoá
    "name",
    "chapter_id",
    "chapter_name",
    "article",
    "content",
]

final_cols   = [c for c in KEEP_COLS if c in legal_df.columns]
legal_final  = legal_df[final_cols].reset_index(drop=True)

OUTPUT_PATHS["legal"].parent.mkdir(parents=True, exist_ok=True)
legal_final.to_parquet(OUTPUT_PATHS["legal"], engine="pyarrow", index=False)

size_mb = OUTPUT_PATHS["legal"].stat().st_size / 1024 / 1024
print(f"✅ Đã lưu: {OUTPUT_PATHS['legal']}")
print(f"   Số hàng    : {len(legal_final):,}")
print(f"   Số cột     : {len(legal_final.columns)}  → {list(legal_final.columns)}")
print(f"   Dung lượng : {size_mb:.1f} MB")

✅ Đã lưu: dataset/processed/legal_dataset_processed.parquet
   Số hàng    : 392,079
   Số cột     : 9  → ['doc_id', 'id', 'ministry', 'type_normalized', 'name', 'chapter_id', 'chapter_name', 'article', 'content']
   Dung lượng : 284.8 MB


### 1.8. Kiểm tra nhanh output

In [9]:
chk = pd.read_parquet(OUTPUT_PATHS["legal"], engine="pyarrow")

print("=" * 55)
print("KIỂM TRA LEGAL OUTPUT")
print("=" * 55)
print(f"Số hàng              : {len(chk):,}")
print(f"Missing content      : {chk['content'].isna().sum() + (chk['content'] == '').sum()}")
print(f"Duplicate doc_id     : {chk['doc_id'].duplicated().sum()}")
print(f"type_normalized (unique): {chk['type_normalized'].nunique()}")
print(f"\nPhân phối type_normalized:")
print(chk["type_normalized"].value_counts().to_string())
print("\n3 hàng mẫu:")
chk[["doc_id", "type_normalized", "ministry", "content"]].head(3)

KIỂM TRA LEGAL OUTPUT
Số hàng              : 392,079
Missing content      : 0
Duplicate doc_id     : 0
type_normalized (unique): 9

Phân phối type_normalized:
type_normalized
THÔNG TƯ        132431
QUYẾT ĐỊNH       85733
NGHỊ ĐỊNH        71176
NGHỊ QUYẾT       59827
LUẬT             31145
PHÁP LỆNH         6798
KHÁC              4859
VĂN BẢN KHÁC        83
KHÔNG RÕ            27

3 hàng mẫu:


,doc_id,type_normalized,ministry,content
0,01/2014/NQLT/CP-UBTƯMTTQVN__Điều_1._Phạm_vi_đi...,NGHỊ QUYẾT,CHÍNH PHỦ - ỦY BAN TRUNG ƯƠNG MẶT TRẬN TỔ QUỐC...,Nghị quyết liên tịch này hướng dẫn phối hợp th...
1,01/2014/NQLT/CP-UBTƯMTTQVN__Điều_2._Nguyên_tắc...,NGHỊ QUYẾT,CHÍNH PHỦ - ỦY BAN TRUNG ƯƠNG MẶT TRẬN TỔ QUỐC...,1. Việc phối hợp hoạt động được thực hiện trên...
2,01/2014/NQLT/CP-UBTƯMTTQVN__Điều_3._Hướng_dẫn_...,NGHỊ QUYẾT,CHÍNH PHỦ - ỦY BAN TRUNG ƯƠNG MẶT TRẬN TỔ QUỐC...,"Ủy ban Trung ương Mặt trận Tổ quốc Việt Nam, Ủ..."


---
## 2. Forms Dataset

**checkData_v2 xác nhận:** Dataset sạch, 0 missing, 0 duplicate.  
→ Chỉ cần normalize text các cột string và lưu processed version.

> Lưu ý: cột `required_fields` kiểu `numpy.ndarray`, `required_fields_parsed` kiểu `list` — **không** normalize, giữ nguyên.

### 2.1. Đọc dữ liệu gốc

In [10]:
forms_df = pd.read_parquet(INPUT_PATHS["forms"], engine="pyarrow")

print(f"Số form : {len(forms_df)}")
print(f"Cột     : {list(forms_df.columns)}")
print("\nKiểu dữ liệu:")
print(forms_df.dtypes)
print("\nMissing:")
print(forms_df.isnull().sum())

Số form : 10
Cột     : ['form_id', 'form_type', 'purpose', 'required_fields', 'template_markdown']

Kiểu dữ liệu:
form_id                 str
form_type               str
purpose                 str
required_fields      object
template_markdown       str
dtype: object

Missing:
form_id              0
form_type            0
purpose              0
required_fields      0
template_markdown    0
dtype: int64


### 2.2. Normalize text (chỉ cột string thuần)

In [11]:
# Xác định cột nào là plain string (bỏ qua list/array/dict)
STRING_COLS = []
for col in forms_df.columns:
    sample = forms_df[col].dropna().head(5)
    if sample.apply(lambda x: isinstance(x, str)).all():
        STRING_COLS.append(col)

print(f"Cột sẽ normalize: {STRING_COLS}")
for col in STRING_COLS:
    forms_df[col] = forms_df[col].apply(normalize_text)
    print(f"  ✅ {col}")

print("\n✅ Normalize text hoàn tất")

Cột sẽ normalize: ['form_id', 'form_type', 'purpose', 'template_markdown']
  ✅ form_id
  ✅ form_type
  ✅ purpose
  ✅ template_markdown

✅ Normalize text hoàn tất


### 2.3. Tạo `doc_id`

In [12]:
# form_id đã unique → dùng trực tiếp
forms_df["doc_id"] = forms_df["form_id"]

print(f"Unique doc_id: {forms_df['doc_id'].nunique()} / {len(forms_df)}")
print("✅ Tất cả doc_id unique" if forms_df['doc_id'].nunique() == len(forms_df) else "⚠️ Có doc_id trùng")

Unique doc_id: 10 / 10
✅ Tất cả doc_id unique


### 2.4. Xử lý cột `required_fields` — serialize numpy array → list

PyArrow có thể gặp vấn đề khi lưu numpy array. Convert sang Python list thuần để đảm bảo tương thích.

In [13]:
def to_python_list(val):
    """Convert numpy array / JSON string → Python list thuần."""
    if isinstance(val, list):
        return val
    if isinstance(val, np.ndarray):
        return val.tolist()
    if isinstance(val, str):
        stripped = val.strip()
        if stripped.startswith("["):
            try:
                parsed = json.loads(stripped)
                return parsed if isinstance(parsed, list) else []
            except (json.JSONDecodeError, ValueError):
                pass
        return [s.strip() for s in stripped.split(",") if s.strip()]
    return []

if "required_fields" in forms_df.columns:
    forms_df["required_fields"] = forms_df["required_fields"].apply(to_python_list)
    print(f"✅ required_fields: numpy array → Python list")
    sample = forms_df["required_fields"].iloc[0]
    print(f"   Mẫu: {type(sample).__name__} với {len(sample)} phần tử")

✅ required_fields: numpy array → Python list
   Mẫu: list với 9 phần tử


### 2.5. Lưu Parquet

In [14]:
# Đặt doc_id lên đầu
cols_order = ["doc_id"] + [c for c in forms_df.columns if c != "doc_id"]
forms_out  = forms_df[cols_order].reset_index(drop=True)

OUTPUT_PATHS["forms"].parent.mkdir(parents=True, exist_ok=True)
forms_out.to_parquet(OUTPUT_PATHS["forms"], engine="pyarrow", index=False)

size_kb = OUTPUT_PATHS["forms"].stat().st_size / 1024
print(f"✅ Đã lưu: {OUTPUT_PATHS['forms']}")
print(f"   Số form    : {len(forms_out)}")
print(f"   Số cột     : {len(forms_out.columns)}")
print(f"   Dung lượng : {size_kb:.1f} KB")

✅ Đã lưu: dataset/processed/forms_dataset_processed.parquet
   Số form    : 10
   Số cột     : 6
   Dung lượng : 11.4 KB


---
## 3. Forms Examples Dataset

**checkData_v2 xác nhận:**
- 0 missing values
- 0 placeholder chưa điền trong `filled_content`
- 0 doc_id trùng
- Phân bố đều: 15 ví dụ/form × 10 forms = 150 ví dụ

→ Chỉ cần normalize text, serialize cột `fields` (dict) để đảm bảo tương thích, và lưu.

### 3.1. Đọc dữ liệu gốc

In [15]:
examples_df = pd.read_parquet(INPUT_PATHS["examples"], engine="pyarrow")

print(f"Số ví dụ : {len(examples_df)}")
print(f"Cột      : {list(examples_df.columns)}")
print("\nKiểu dữ liệu:")
print(examples_df.dtypes)
print("\nMissing:")
print(examples_df.isnull().sum())

Số ví dụ : 150
Cột      : ['form_id', 'form_type', 'example_id', 'scenario', 'fields', 'filled_content', 'doc_id']

Kiểu dữ liệu:
form_id              str
form_type            str
example_id           str
scenario             str
fields            object
filled_content       str
doc_id               str
dtype: object

Missing:
form_id           0
form_type         0
example_id        0
scenario          0
fields            0
filled_content    0
doc_id            0
dtype: int64


### 3.2. Normalize text (cột string thuần)

In [16]:
# Xác định cột plain string
EX_STRING_COLS = []
for col in examples_df.columns:
    sample = examples_df[col].dropna().head(5)
    if sample.apply(lambda x: isinstance(x, str)).all():
        EX_STRING_COLS.append(col)

print(f"Cột sẽ normalize: {EX_STRING_COLS}")
for col in EX_STRING_COLS:
    examples_df[col] = examples_df[col].apply(normalize_text)
    print(f"  ✅ {col}")

print("\n✅ Normalize text hoàn tất")

Cột sẽ normalize: ['form_id', 'form_type', 'example_id', 'scenario', 'filled_content', 'doc_id']
  ✅ form_id
  ✅ form_type
  ✅ example_id
  ✅ scenario
  ✅ filled_content
  ✅ doc_id

✅ Normalize text hoàn tất


### 3.3. Kiểm tra lại `filled_content` — đảm bảo không còn placeholder sau normalize

In [17]:
if "filled_content" in examples_df.columns:
    remaining = examples_df["filled_content"].apply(
        lambda x: re.findall(r"\{\{[A-Z_]+\}\}", str(x)) if isinstance(x, str) else []
    )
    bad = remaining.apply(len) > 0
    if bad.sum() == 0:
        print(f"✅ Tất cả {len(examples_df)} ví dụ đã điền đầy đủ — không còn placeholder.")
    else:
        print(f"⚠️  {bad.sum()} ví dụ còn placeholder chưa điền:")
        print(examples_df.loc[bad, ["example_id", "form_type"]].to_string())

✅ Tất cả 150 ví dụ đã điền đầy đủ — không còn placeholder.


### 3.4. Xử lý cột `fields` — đảm bảo tương thích Parquet

Cột `fields` chứa dict Python với các value có thể là `None` hoặc string.  
Đảm bảo serialize đúng để PyArrow không gặp lỗi mixed types.

In [18]:
if "fields" in examples_df.columns:
    # Kiểm tra kiểu dữ liệu thực tế của cột
    sample = examples_df["fields"].iloc[0]
    print(f"Kiểu dữ liệu cột 'fields': {type(sample).__name__}")

    if isinstance(sample, str):
        # Nếu là JSON string, parse lại
        def safe_parse_dict(val):
            if isinstance(val, dict):
                return val
            if isinstance(val, str):
                try:
                    return json.loads(val)
                except Exception:
                    return {}
            return val if val is not None else {}
        examples_df["fields"] = examples_df["fields"].apply(safe_parse_dict)
        print("  → Đã parse JSON string → dict")

    # Serialize về JSON string để Parquet lưu an toàn
    examples_df["fields_json"] = examples_df["fields"].apply(
        lambda x: json.dumps(x, ensure_ascii=False) if isinstance(x, dict) else "{}"
    )
    print(f"  ✅ Đã tạo cột 'fields_json' (JSON string) để lưu Parquet")
    print(f"     Mẫu: {examples_df['fields_json'].iloc[0][:100]}...")

Kiểu dữ liệu cột 'fields': dict
  ✅ Đã tạo cột 'fields_json' (JSON string) để lưu Parquet
     Mẫu: {"CAC_VAN_DE_CAN_LUU_Y": null, "CAN_CU_PHAP_LY_1": "Luật Tổ chức chính quyền địa phương ngày 19 thán...


### 3.5. Xác nhận `doc_id`

In [19]:
# doc_id đã tồn tại và unique (checkData xác nhận)
id_col = "doc_id" if "doc_id" in examples_df.columns else "example_id"
total  = len(examples_df)
unique = examples_df[id_col].nunique()

print(f"Cột ID: '{id_col}'")
print(f"Unique : {unique} / {total}")
if unique == total:
    print("✅ Tất cả doc_id unique")
    # Đảm bảo có cột doc_id chuẩn
    if id_col != "doc_id":
        examples_df["doc_id"] = examples_df[id_col]
else:
    print("⚠️ Có doc_id trùng — kiểm tra lại")

Cột ID: 'doc_id'
Unique : 150 / 150
✅ Tất cả doc_id unique


### 3.6. Lưu Parquet

In [20]:
# Chọn cột output — giữ fields_json thay cho fields (dict) nếu đã tạo
KEEP_EX_COLS = ["doc_id", "form_id", "form_type", "example_id",
                "scenario", "fields_json", "filled_content"]

final_ex_cols = [c for c in KEEP_EX_COLS if c in examples_df.columns]
# Thêm các cột còn lại chưa có trong KEEP_EX_COLS
extra_cols = [c for c in examples_df.columns if c not in final_ex_cols and c != "fields"]
final_ex_cols = final_ex_cols + extra_cols

examples_out = examples_df[final_ex_cols].reset_index(drop=True)

OUTPUT_PATHS["examples"].parent.mkdir(parents=True, exist_ok=True)
examples_out.to_parquet(OUTPUT_PATHS["examples"], engine="pyarrow", index=False)

size_kb = OUTPUT_PATHS["examples"].stat().st_size / 1024
print(f"✅ Đã lưu: {OUTPUT_PATHS['examples']}")
print(f"   Số ví dụ  : {len(examples_out)}")
print(f"   Số cột    : {len(examples_out.columns)}  → {list(examples_out.columns)}")
print(f"   Dung lượng: {size_kb:.1f} KB")

✅ Đã lưu: dataset/processed/forms_examples_dataset_processed.parquet
   Số ví dụ  : 150
   Số cột    : 7  → ['doc_id', 'form_id', 'form_type', 'example_id', 'scenario', 'fields_json', 'filled_content']
   Dung lượng: 249.9 KB


---
## 4. Tổng kết & Báo cáo

In [21]:
print("=" * 70)
print("         BÁO CÁO PREPROCESSING")
print("=" * 70)

datasets = {
    "legal_dataset"         : (INPUT_PATHS["legal"],    OUTPUT_PATHS["legal"]),
    "forms_dataset"         : (INPUT_PATHS["forms"],    OUTPUT_PATHS["forms"]),
    "forms_examples_dataset": (INPUT_PATHS["examples"], OUTPUT_PATHS["examples"]),
}

for name, (inp, out) in datasets.items():
    if not out.exists():
        print(f"\n[{name}]  ❌ Output chưa tạo được")
        continue

    df_in  = pd.read_parquet(inp, engine="pyarrow")
    df_out = pd.read_parquet(out, engine="pyarrow")

    n_in   = len(df_in)
    n_out  = len(df_out)
    removed = n_in - n_out
    dup_id  = df_out["doc_id"].duplicated().sum() if "doc_id" in df_out.columns else "N/A"

    print(f"\n[{name}]")
    print(f"  Input      : {inp}  ({n_in:,} hàng)")
    print(f"  Output     : {out}  ({n_out:,} hàng)")
    print(f"  Đã loại bỏ: {removed:,} hàng  ({removed/n_in*100:.1f}%)")
    print(f"  Duplicate doc_id: {dup_id}")
    print(f"  Cột output : {list(df_out.columns)}")

print("\n" + "=" * 70)
print("✅ Data cleaning hoàn tất.")
print("   Bước tiếp theo: Chunking (notebook riêng)")
print("=" * 70)

         BÁO CÁO PREPROCESSING

[legal_dataset]
  Input      : ../dataset/raw/legal_dataset.parquet  (515,188 hàng)
  Output     : dataset/processed/legal_dataset_processed.parquet  (392,079 hàng)
  Đã loại bỏ: 123,109 hàng  (23.9%)
  Duplicate doc_id: 0
  Cột output : ['doc_id', 'id', 'ministry', 'type_normalized', 'name', 'chapter_id', 'chapter_name', 'article', 'content']

[forms_dataset]
  Input      : ../dataset/raw/forms_dataset.parquet  (10 hàng)
  Output     : dataset/processed/forms_dataset_processed.parquet  (10 hàng)
  Đã loại bỏ: 0 hàng  (0.0%)
  Duplicate doc_id: 0
  Cột output : ['doc_id', 'form_id', 'form_type', 'purpose', 'required_fields', 'template_markdown']

[forms_examples_dataset]
  Input      : ../dataset/raw/forms_examples_dataset.parquet  (150 hàng)
  Output     : dataset/processed/forms_examples_dataset_processed.parquet  (150 hàng)
  Đã loại bỏ: 0 hàng  (0.0%)
  Duplicate doc_id: 0
  Cột output : ['doc_id', 'form_id', 'form_type', 'example_id', 'scenario', 'f